# Demonstração da falha do modelo tradicional ao injetar documentos obsoletos
O objetivo do **Notebook 02** é provar que a busca vetorial (similaridade de cosseno) resolve o problema de encontrar textos parecidos, mas não compreende o contexto.

A nossa base tem **duas versões da regra de Eletrônicos**: a de 2021 (restritiva: caixa lacrada, 7 dias) e a atualização de 2024 (flexível: 30 dias, mesmo com a caixa aberta). As duas são semanticamente quase idênticas, então o Qdrant devolve ambas. O LLM recebe regras contraditórias e **não tem como saber qual é a vigente**.

## 1. Configuração e Conexão (Setup)
No primeiro bloco, conectamos aos serviços que subimos no Docker e recuperamos a coleção já populada no Qdrant.

In [1]:
import ollama
from qdrant_client import QdrantClient

EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
LLM_MODEL = "llama3.1"
TOP_K = 5  # mesmo valor do Notebook 03, para a comparação ser justa

In [2]:
# Conectar ao Qdrant e carregar a coleção existente
client = QdrantClient(url="http://localhost:6333")
collection_name = "portfolio_agentic_rag"

## 2. O Problema: Busca Cega (Naive Retrieval)
Aqui trazemos os 5 resultados mais "similares" matematicamente.

Imprimimos a data de importação só para *nós* enxergarmos o problema. O LLM recebe apenas o texto, como em um RAG tradicional.

In [3]:
pergunta = "Comprei um laptop ontem, abri a caixa para olhar e me arrependi. Qual o prazo e a regra para devolver?"

# Gerar o embedding da pergunta (prefixo de busca do modelo Nomic)
vetor_pergunta = ollama.embed(
    model=EMBEDDING_MODEL,
    input=f"search_query: {pergunta}"
)["embeddings"][0]

# Buscar os trechos mais similares no Qdrant
resultados = client.query_points(
    collection_name=collection_name,
    query=vetor_pergunta,
    limit=TOP_K
).points

print("--- DOCUMENTOS RECUPERADOS PELO QDRANT (SEM FILTRO) ---")
for res in resultados:
    print(f"Score: {res.score:.4f} | Importado em {res.payload['data_importacao']} | {res.payload['texto'][:70]}...")

contexto_bruto = "\n\n".join(res.payload["texto"] for res in resultados)

--- DOCUMENTOS RECUPERADOS PELO QDRANT (SEM FILTRO) ---
Score: 0.7470 | Importado em 2021-03-10 | 5. Regras para Eletrônicos e Informática (Laptops, Notebooks, Celulare...
Score: 0.7395 | Importado em 2024-03-01 | 5. Regras para Eletrônicos e Informática (Laptops, Notebooks, Celulare...
Score: 0.6013 | Importado em 2021-03-10 | 4. Regras para Brinquedos e Artigos Infantis
Brinquedos possuem prazo ...
Score: 0.5908 | Importado em 2021-03-10 | 3. Regras para Móveis e Decoração
O prazo de arrependimento para móvei...
Score: 0.5796 | Importado em 2021-03-10 | 8. Análise Técnica e Reembolso
O produto devolvido passa por análise t...


## 3. A Geração Falha (LLM Chain)
Nesta etapa, jogamos tudo no prompt. Usamos `temperature=0` para que o resultado seja reproduzível e o **mesmo prompt** do Notebook 03, assim a única diferença entre os dois notebooks é o filtro.

In [4]:
PROMPT_SISTEMA = """Você é um assistente de atendimento ao cliente da Loja Demo.
Responda à pergunta do usuário baseando-se APENAS no contexto fornecido.

CONTEXTO:
{contexto}
"""

resposta_naive = ollama.chat(
    model=LLM_MODEL,
    options={"temperature": 0},
    messages=[
        {"role": "system", "content": PROMPT_SISTEMA.format(contexto=contexto_bruto)},
        {"role": "user", "content": pergunta}
    ]
)

In [5]:
print("\n--- RESPOSTA DO NAIVE RAG ---")
print(resposta_naive["message"]["content"])


--- RESPOSTA DO NAIVE RAG ---
Lamento ouvir que você se arrependeu da compra!

De acordo com as nossas regras, se você abriu a caixa do laptop para olhar, mesmo que apenas para conferência visual, você perde o direito à devolução por arrependimento.

No entanto, você pode considerar a opção de devolução por defeito, desde que o laptop apresente algum problema técnico. Nesse caso, você precisará enviar o produto para análise técnica no Centro de Distribuição e, se a devolução for aprovada, você receberá o reembolso de acordo com as nossas regras.

Se você tiver alguma dúvida ou precisar de mais informações, por favor, entre em contato conosco. Estamos aqui para ajudar!


### O Problema do Naive RAG: Contaminação por Regra Obsoleta

O Qdrant fez o trabalho dele: as duas versões da regra de Eletrônicos ficaram no topo do ranking. Mas o LLM recebeu duas instruções contraditórias ("caixa aberta = perde o direito" e "caixa aberta = devolução aceita em 30 dias") e **aplicou a regra de 2021, que já foi substituída**.

O cliente teve um direito vigente negado. No mundo real isso gera reclamação no Procon, retrabalho no SAC e perda de confiança.

Observe também que o Llama 3.1 tende a escolher a regra **mais restritiva** quando encontra um conflito. Não existe critério temporal nessa escolha; é só um viés do modelo. Quando a regra nova for a restritiva, ele vai "acertar por acaso", o que torna o erro ainda mais difícil de detectar em testes.

**A Solução:** a informação que resolve o conflito (a data de importação) existe, mas está no metadado, e a busca vetorial não sabe que dois trechos falam *da mesma regra*. É aqui que nasce a necessidade de um **Agente de Relevância (Laya)**: uma camada que identifica qual regra cada trecho define e descarta as versões substituídas antes que elas cheguem ao LLM gerador.